# Synthetic Data Generation for Embedding Training with Distilabel

[![Open In Colab](https://img.shields.io/badge/Open%20In-Colab-blue?style=for-the-badge&logo=google-colab)](https://colab.research.google.com/github/dnth/rag-datakit/blob/main/nbs/00_distilabel-synthetic-data-generation.ipynb)
[![Open In Kaggle](https://img.shields.io/badge/Open%20In-Kaggle-blue?style=for-the-badge&logo=kaggle)](https://kaggle.com/kernels/welcome?src=https://github.com/dnth/rag-datakit/blob/main/nbs/00_distilabel-synthetic-data-generation.ipynb)

This notebook demonstrates how to use `distilabel` to generate synthetic training data for customized embedding models. We'll work with job descriptions from Singapore's SkillsFuture Framework to create positive and negative query pairs that can be used to train embedding models for better job matching and semantic search capabilities.

## What you'll learn:
- How to load and prepare datasets for synthetic data generation
- Creating positive and negative query pairs using LLMs
- Building distilabel pipelines for automated data generation
- Publishing synthetic datasets to Hugging Face Hub

On Google Colab you might need to uninstall the existing packages due to conflicting versions.

In [1]:
# !pip uninstall -y transformers torch torchvision

## Installation

Install the rag-datakit package which includes all necessary dependencies including distilabel, transformers, and dataset utilities. Uncomment the cell below to install if you haven't already.

In [2]:
# !pip install git+https://github.com/dnth/rag-datakit.git

## Login Hugging Face

Login your Hugging Face account as we will be uploading the output to Hugging Face.

In [3]:
# !hf auth login

## Dataset Loading and Inspection

We'll work with the Singapore Skills Framework (SSF) dataset, which contains job roles and descriptions across various sectors. This dataset is ideal for training embedding models for job matching applications.

**Dataset source:** [Skills Frameworks Singapore](https://jobsandskills.skillsfuture.gov.sg/frameworks/skills-frameworks)  
**Hugging Face repo:** `dnth/ssf-dataset`

The dataset contains structured information about:
- **Sector**: Industry sector (e.g., Accountancy, Technology)
- **Track**: Specialization within the sector (e.g., Assurance, Business Valuation)
- **Job Role**: Specific job title
- **Job Role Description**: Detailed description of responsibilities and requirements
- **Performance Expectation**: Standards and compliance requirements

Let's load and examine the dataset structure:

In [4]:
from datasets import load_dataset

dataset = load_dataset("dnth/ssf-dataset")

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Sector', 'Track', 'Job Role', 'Job Role Description', 'Performance Expectation'],
        num_rows: 1885
    })
})

In [6]:
dataset['train'][0]

{'Sector': 'Accountancy',
 'Track': 'Assurance',
 'Job Role': 'Audit Associate / Audit Assistant Associate',
 'Job Role Description': 'The Audit Associate/Audit Assistant Associate undertakes specific stages of audit work under supervision. He/She begins to appreciate the underlying principles behind the tasks assigned to him as part of the audit plan. He is also able to make adjustments to the application of skills to improve the work tasks or solve non-complex issues. The Audit Associate/Audit Assistant Associate operates in a structured work environment. He is able to build relationships, work in a team and identify ethical issues with reference to the code of professional conduct and ethics. He is able to select and apply from a range of known solutions to familiar problems and takes responsibility for his own learning and performance. He is a trustworthy and meticulous individual.',
 'Performance Expectation': 'In accordance with: Singapore Standards on Auditing, Ethics Pronouncem

## Synthetic Data Generation Setup

To train effective embedding models, we need to create training triplets consisting of:
- **Anchor**: The original job role description
- **Positive**: A paraphrased or similar job description (semantically similar)
- **Negative**: A different job description (semantically dissimilar)

We'll use a Large Language Model (LLM) to generate these positive and negative examples automatically. This approach allows us to:
1. Create diverse paraphrases of job descriptions
2. Generate realistic negative examples from different roles/industries
3. Scale up our training data efficiently

### LLM Configuration
We support both local models (via Transformers) and API-based models (like OpenAI). For this example, we'll use a local Qwen model, but you can switch to OpenAI by uncommenting the appropriate section.

**For OpenAI API**: Make sure you have your API key in a `.env` file:
```
OPENAI_API_KEY=sk-proj-...
```

In [7]:
import os
from distilabel.models import OpenAILLM, TransformersLLM

# llm = TransformersLLM(
#     model="Qwen/Qwen3-4B-Instruct-2507",
#     device_map="auto",
#     torch_dtype="float16",
# )

llm = OpenAILLM(
    model="gpt-4.1-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
)

In [8]:
# context = """
# The text is a job description from the Singapore SkillsFuture Framework. Your task is to generate realistic job descriptions from the provided description.

# For the positive query, generate a realistic description for this role. Focus on creating variations that capture the essence of the role in different words, as if written by different people or organizations posting similar jobs.

# For negative descriptions you are allowed to choose from the following strategies

# 1. Same industry, different seniority level (Senior → Junior or Vice versa)
# 2. Same industry, different function (Business Valuation → Risk Management)
# 3. Similar skills, different domain (Financial Analysis in Banking vs Healthcare)
# 4. Same title, different industry context
# 5. Create roles that sound similar but serve different business needs

# The query should always include the job role. Start the description with The <job role>.

# """

context = """
## Task Overview
You are tasked with generating realistic job descriptions based on Singapore SkillsFuture Framework job descriptions. Your goal is to create both positive and negative examples for training a retrieval model.

## Input Format
You will receive a job description from the Singapore SkillsFuture Framework containing:
- Job title (e.g., Audit Associate/Audit Assistant Associate)
- Role responsibilities and duties
- Work environment and supervision structure
- Required skills and attributes
- Professional conduct expectations

## Output Requirements

### Positive Description
Generate using any of the following strategies:
- Captures the essence of the original role in different words
- Maintains the same seniority level and core responsibilities  
- Uses varied terminology while preserving the role's fundamental nature
- Reads as if written by a different organization posting a similar position
- Starts with "The [job role]"

### Negative Descriptions  


Generate using ONE of the following strategies:

#### 1. Easy Negative - Different Function, Same Industry
- Change the core function while keeping the same industry
- Use completely different skill requirements
- Maintain similar professional context
- Example: Audit Associate → Tax Associate

#### 2. Medium Negative - Same Industry, Different Seniority
- Significantly change the responsibility level (Junior → Senior or vice versa)
- Alter supervision structure (individual contributor → manager)
- Modify years of experience and decision-making authority
- Example: Audit Associate → Senior Audit Manager

#### 3. Hard Negative - Same Skills, Different Domain  
- Transfer core skills to a completely different industry
- Maintain similar analytical/technical requirements
- Change regulatory environment and business context
- Example: Audit Associate → Compliance Associate (Banking)

#### 4. Hard Negative - Geographic/Regulatory Variation
- Same role with different regulatory requirements
- Vary market maturity and business practices
- Include cross-border or international elements
- Change compliance frameworks and standards

#### 5. Very Hard Negative - Hybrid Role Confusion
- Combine responsibilities from multiple distinct roles
- Create plausible but incorrect role combinations
- Mix strategic and tactical responsibilities inappropriately
- Include overlapping but different skill requirements

Starts with "The [job role]"

## Quality Guidelines

### For All Descriptions:
- Start each description with "The [job role]"
- Maintain professional, realistic language
- Ensure descriptions are plausible and well-written
- Include specific responsibilities, skills, and requirements
- Vary sentence structure and terminology naturally

### For Negative Examples:
- **Lexical Overlap**: Include similar keywords but different intent
- **Semantic Similarity**: Create roles that sound related but serve different purposes
- **Contextual Differences**: Same skills applied in different business contexts
- **Responsibility Scope**: Vary breadth and depth of responsibilities significantly

### Difficulty Gradation:
- **Easy negatives**: Clearly different functions/industries
- **Medium negatives**: Same industry, different seniority or function
- **Hard negatives**: Subtle contextual differences that require careful analysis
- **Very hard negatives**: Realistic but incorrect combinations that could fool initial screening

## Example Structure

**Input**: [Singapore SkillsFuture job description]

**Positive Description**: 
The [Job Role] [realistic variation maintaining core essence]...

**Negative Descriptions**:

The [Job Role] [negative example description]...
"""

from distilabel.pipeline import Pipeline
from distilabel.steps import LoadDataFromHub
from distilabel.steps.tasks import GenerateSentencePair

with Pipeline(name="generate") as pipeline:
    load_dataset = LoadDataFromHub(
        # num_examples=10,  # Limit to 10 examples for demo - increase for production datasets
        use_cache=False,  # Disable caching to ensure fresh data generation each run
        output_mappings={"Job Role Description": "anchor"},  # Map original column to 'anchor' for triplet generation
    )
    generate_retrieval_pairs_easy = GenerateSentencePair(
        name="easy_triplets",
        triplet=True,  # Generate anchor-positive-negative triplets for embedding training
        hard_negative=False,  # Use easier negatives rather than hard negatives
        action="paraphrase",  # Focus on paraphrasing for positive examples
        llm=llm,  # Use the LLM configured above (local Qwen or OpenAI)
        input_batch_size=10,  # Process 10 examples at once for efficiency
        context=context,  # Provide the context instructions for generation quality
    )
    generate_retrieval_pairs_hard = GenerateSentencePair(
        name="hard_triplets",
        triplet=True,  
        hard_negative=True,  
        action="paraphrase",  
        llm=llm,  
        input_batch_size=10,  
        context=context,  
    )

    load_dataset.connect(generate_retrieval_pairs_easy, generate_retrieval_pairs_hard)

## Pipeline Configuration

Now we'll set up a distilabel pipeline to automatically generate positive and negative examples. The pipeline consists of:

1. **LoadDataFromHub**: Loads our SSF dataset from Hugging Face
2. **GenerateSentencePair**: Uses the LLM to create positive/negative pairs

### Context for LLM Generation
We provide specific instructions to the LLM for generating realistic job descriptions:
- **Positive examples**: Paraphrases that maintain the essence of the original role
- **Negative examples**: Use strategies like different seniority levels, functions, or industries

The pipeline is configured to generate triplets (anchor, positive, negative) for embedding training. The first run may take more time because the model is being downloaded if you are using the Transformers LLM.

In [9]:
distiset = pipeline.run(
    use_cache=False,
    parameters={
        load_dataset.name: {
            "repo_id": "dnth/ssf-dataset",
            "split": "train",
        },
        "easy_triplets": {
            "llm": {"generation_kwargs": {"temperature": 0.6, "max_new_tokens": 512}}
        },
        "hard_triplets": {
            "llm": {"generation_kwargs": {"temperature": 0.7, "max_new_tokens": 512}}
        },
    }
)

[09/09/25 19:46:54] INFO     ['distilabel.pipeline'] 📝 Pipeline data will be written to               ]8;id=508954;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=752760;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py#1015\1015]8;;\
                             '/home/dnth/.cache/distilabel/pipelines/generate/3969fa9affab4d8b8bf2f804             
                             bce1882fe95d8fcc/executions/fc2c8c66032067fda24d55e04ff63e9b106f07ef/data             
                             /steps_outputs'                                                                       

                    INFO     ['distilabel.pipeline'] ⌛ The steps of the pipeline will be loaded in    ]8;id=685196;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=712699;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py#1046\1046]8;;\
                             stages:                                                                               
                              * Legend: 🚰 GeneratorStep 🌐 GlobalStep 🔄 Step                                     
                              * Stage 0:                                                                           
                                - 🚰 'load_data_from_hub_0'                                                        
                                - 🔄 'easy_triplets'                                                               
                                - 🔄 'hard_triplets'                                                               

                    INFO     ['distilabel.pipeline'] ⏳ Waiting for all the steps of stage 0 to        ]8;id=858808;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=579908;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py#1382\1382]8;;\
                             load...                                                                               

[09/09/25 19:46:57] INFO     ['distilabel.pipeline'] ⏳ Steps from stage 0 loaded: 2/3                 ]8;id=385245;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=30013;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py#1418\1418]8;;\
                              * 'load_data_from_hub_0' replicas: 0/1                                               
                              * 'easy_triplets' replicas: 1/1                                                      
                              * 'hard_triplets' replicas: 1/1                                                      

[09/09/25 19:46:59] INFO     ['distilabel.pipeline'] ⏳ Steps from stage 0 loaded: 3/3                 ]8;id=35233;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=732880;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py#1418\1418]8;;\
                              * 'load_data_from_hub_0' replicas: 1/1                                               
                              * 'easy_triplets' replicas: 1/1                                                      
                              * 'hard_triplets' replicas: 1/1                                                      

...

                    INFO     ['distilabel.step.hard_triplets'] 🏁 Finished running step         ]8;id=787840;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=3525;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/step_wrapper.py#129\129]8;;\
                             'hard_triplets' (replica ID: 0)                                                       

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

## Running the Pipeline

Execute the pipeline with specific parameters:
- **num_examples**: Limit to 10 examples for demonstration (increase for production)
- **temperature**: Controls randomness in LLM generation (0.7 for creative but consistent outputs)
- **max_new_tokens**: Maximum length of generated text

The pipeline will process each job description and generate corresponding positive and negative examples.

In [10]:
distiset

Distiset({
    easy_triplets: DatasetDict({
        train: Dataset({
            features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
            num_rows: 1885
        })
    })
    hard_triplets: DatasetDict({
        train: Dataset({
            features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
            num_rows: 1885
        })
    })
})

## Results Inspection

The pipeline generates a `Distiset` object containing our synthetic dataset. Let's examine the structure and content of the generated data:

### Dataset Structure
The output includes the original fields plus new generated columns:
- **anchor**: Original job role description
- **positive**: LLM-generated paraphrase (semantically similar)
- **negative**: LLM-generated different job description (semantically dissimilar)
- **distilabel_metadata**: Generation metadata and token usage
- **model_name**: LLM model used for generation

In [11]:
distiset["easy_triplets"]["train"][0]

{'Sector': 'Accountancy',
 'Track': 'Assurance',
 'Job Role': 'Audit Associate / Audit Assistant Associate',
 'anchor': 'The Audit Associate/Audit Assistant Associate undertakes specific stages of audit work under supervision. He/She begins to appreciate the underlying principles behind the tasks assigned to him as part of the audit plan. He is also able to make adjustments to the application of skills to improve the work tasks or solve non-complex issues. The Audit Associate/Audit Assistant Associate operates in a structured work environment. He is able to build relationships, work in a team and identify ethical issues with reference to the code of professional conduct and ethics. He is able to select and apply from a range of known solutions to familiar problems and takes responsibility for his own learning and performance. He is a trustworthy and meticulous individual.',
 'Performance Expectation': 'In accordance with: Singapore Standards on Auditing, Ethics Pronouncements in Singap

In [12]:
distiset["easy_triplets"]["train"].to_pandas()

,Sector,Track,Job Role,anchor,Performance Expectation,positive,negative,distilabel_metadata,model_name
0,Accountancy,Assurance,Audit Associate / Audit Assistant Associate,The Audit Associate/Audit Assistant Associate ...,In accordance with: Singapore Standards on Aud...,The Audit Associate/Audit Assistant Associate ...,The Tax Associate handles tax preparation and ...,{'raw_input_easy_triplets': [{'content': 'Your...,gpt-4.1-mini
1,Accountancy,Assurance,Audit Manager,The Audit Senior Manager/Audit Manager manages...,In accordance with: Singapore Standards on Aud...,The Audit Manager is responsible for overseein...,The Tax Manager leads the tax compliance and a...,{'raw_input_easy_triplets': [{'content': 'Your...,gpt-4.1-mini
2,Accountancy,Assurance,Audit Partner / Audit Director,The Audit Partner/Audit Director is a transfor...,In accordance with: Singapore Standards on Aud...,The Audit Partner/Audit Director serves as a v...,The Compliance Partner/Compliance Director lea...,{'raw_input_easy_triplets': [{'content': 'Your...,gpt-4.1-mini
3,Accountancy,Assurance,Audit Senior,The Audit Senior is expected to team lead vari...,In accordance with: Singapore Standards on Aud...,The Audit Senior leads audit projects of varyi...,The Tax Associate focuses on preparing and fil...,{'raw_input_easy_triplets': [{'content': 'Your...,gpt-4.1-mini
4,Accountancy,Business Valuation,Business Valuation Associate / Business Valuat...,The Business Valuation Associate/Business Valu...,In accordance with the International Valuation...,The Business Valuation Associate/Executive pla...,The Tax Associate is responsible for preparing...,{'raw_input_easy_triplets': [{'content': 'Your...,gpt-4.1-mini
...,...,...,...,...,...,...,...,...,...
1880,Workplace Safety and Health,Operational Control,Workplace Safety and Health Manager,The WSH Manager is responsible for reviewing W...,In accordance with: Workplace Safety and Healt...,The WSH Manager oversees the evaluation and up...,The WSH Coordinator is responsible for support...,{'raw_input_easy_triplets': [{'content': 'Your...,gpt-4.1-mini
1881,Workplace Safety and Health,Operational Control,Workplace Safety and Health Officer,The WSH Officer is responsible for developing ...,In accordance with: Workplace Safety and Healt...,The WSH Officer is tasked with designing and o...,The WSH Training Coordinator develops and deli...,{'raw_input_easy_triplets': [{'content': 'Your...,gpt-4.1-mini
1882,Workplace Safety and Health,Operational Control,Workplace Safety and Health Supervisor,The Workplace Safety and Health (WSH) Supervis...,In accordance with: Workplace Safety and Healt...,The Workplace Safety and Health (WSH) Supervis...,The Workplace Safety and Health (WSH) Coordina...,{'raw_input_easy_triplets': [{'content': 'Your...,gpt-4.1-mini
1883,Workplace Safety and Health,System Audit,Lead Workplace Safety and Health Auditor,The Lead Workplace Safety and Health (WSH) Aud...,In accordance with: Workplace Safety and Healt...,The Lead Workplace Safety and Health (WSH) Aud...,The Lead Environmental Compliance Officer mana...,{'raw_input_easy_triplets': [{'content': 'Your...,gpt-4.1-mini


In [13]:
distiset["hard_triplets"]["train"].to_pandas()

,Sector,Track,Job Role,anchor,Performance Expectation,positive,negative,distilabel_metadata,model_name
0,Accountancy,Assurance,Audit Associate / Audit Assistant Associate,The Audit Associate/Audit Assistant Associate ...,In accordance with: Singapore Standards on Aud...,The Audit Associate/Audit Assistant Associate ...,The Audit Associate/Audit Assistant Associate ...,{'raw_input_hard_triplets': [{'content': 'Your...,gpt-4.1-mini
1,Accountancy,Assurance,Audit Manager,The Audit Senior Manager/Audit Manager manages...,In accordance with: Singapore Standards on Aud...,The Audit Manager is responsible for overseein...,The Audit Senior Consultant leads the preparat...,{'raw_input_hard_triplets': [{'content': 'Your...,gpt-4.1-mini
2,Accountancy,Assurance,Audit Partner / Audit Director,The Audit Partner/Audit Director is a transfor...,In accordance with: Singapore Standards on Aud...,The Audit Partner/Audit Director serves as a v...,The Audit Partner/Audit Director leads cross-b...,{'raw_input_hard_triplets': [{'content': 'Your...,gpt-4.1-mini
3,Accountancy,Assurance,Audit Senior,The Audit Senior is expected to team lead vari...,In accordance with: Singapore Standards on Aud...,The Audit Senior is responsible for leading au...,The Audit Manager oversees the preparation and...,{'raw_input_hard_triplets': [{'content': 'Your...,gpt-4.1-mini
4,Accountancy,Business Valuation,Business Valuation Associate / Business Valuat...,The Business Valuation Associate/Business Valu...,In accordance with the International Valuation...,The Business Valuation Executive is responsibl...,The Business Compliance Executive is tasked wi...,{'raw_input_hard_triplets': [{'content': 'Your...,gpt-4.1-mini
...,...,...,...,...,...,...,...,...,...
1880,Workplace Safety and Health,Operational Control,Workplace Safety and Health Manager,The WSH Manager is responsible for reviewing W...,In accordance with: Workplace Safety and Healt...,The WSH Manager oversees the evaluation and co...,The WSH Coordinator is responsible for assisti...,{'raw_input_hard_triplets': [{'content': 'Your...,gpt-4.1-mini
1881,Workplace Safety and Health,Operational Control,Workplace Safety and Health Officer,The WSH Officer is responsible for developing ...,In accordance with: Workplace Safety and Healt...,The WSH Officer oversees the creation and impl...,The WSH Coordinator manages employee wellness ...,{'raw_input_hard_triplets': [{'content': 'Your...,gpt-4.1-mini
1882,Workplace Safety and Health,Operational Control,Workplace Safety and Health Supervisor,The Workplace Safety and Health (WSH) Supervis...,In accordance with: Workplace Safety and Healt...,The Workplace Safety and Health (WSH) Supervis...,The Workplace Safety and Health (WSH) Coordina...,{'raw_input_hard_triplets': [{'content': 'Your...,gpt-4.1-mini
1883,Workplace Safety and Health,System Audit,Lead Workplace Safety and Health Auditor,The Lead Workplace Safety and Health (WSH) Aud...,In accordance with: Workplace Safety and Healt...,The Lead Workplace Safety and Health (WSH) Aud...,The Lead Environmental Compliance Auditor is r...,{'raw_input_hard_triplets': [{'content': 'Your...,gpt-4.1-mini


## Publishing to Hugging Face Hub

Finally, we can push our synthetic dataset to Hugging Face Hub for sharing and future use. This makes the dataset easily accessible for training embedding models or other downstream tasks.

The dataset will be uploaded with all the generated triplets and metadata, ready for use in embedding training pipelines.

In [14]:
distiset.push_to_hub("dnth/ssf-dataset-synthetic-v3")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  11%|#1        |  528kB / 4.74MB            

...

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  11%|#         |  528kB / 4.92MB            

README.md:   0%|          | 0.00/973 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

## Customization Options

This notebook provides several parameters that can be modified to customize the synthetic data generation process for your specific use case:

### 1. Large Language Model (LLM) Selection
- **Local Models**: Use `TransformersLLM` with different models:
  - `Qwen/Qwen3-4B-Instruct-2507` (current)
  - `microsoft/DialoGPT-medium`
  - `meta-llama/Llama-2-7b-chat-hf`
- **API Models**: Switch to `OpenAILLM` with models like:
  - `gpt-4o-mini` (cost-effective)
  - `gpt-4o` (higher quality)
  - `gpt-3.5-turbo` (faster, cheaper)

See more [here](https://huggingface.co/models?pipeline_tag=text-generation&num_parameters=min:0,max:6B&sort=trending).

### 2. LLM Generation Parameters
- **temperature**: Controls randomness (0.1 = conservative, 1.0 = creative)
- **max_new_tokens**: Maximum length of generated text (128-512 typical)
- **top_p**: Nucleus sampling parameter for diversity control
- **repetition_penalty**: Prevents repetitive text generation

### 3. GenerateSentencePair Configuration
- **hard_negative**: 
  - `False`: Generates easier negatives for stable training
  - `True` (current): Creates harder negatives that are more challenging to distinguish
- **action**: 
  - `"paraphrase"` (current): Focus on paraphrasing
  - `"semantically-similar"`: More creative generation
- **input_batch_size**: Adjust based on your hardware capabilities (1-50)

Read more [here](https://github.com/argilla-io/distilabel/blob/main/docs/sections/pipeline_samples/tutorials/GenerateSentencePair.ipynb).

### 4. Context Prompt Engineering
The `context` variable can be modified to:
- Add industry-specific instructions
- Include different negative generation strategies
- Specify output format requirements
- Add quality control guidelines
- Target specific job domains or levels

### 5. Dataset Parameters
- **num_examples**: Scale from 10 (demo) to thousands (production)


These parameters allow you to fine-tune the generation process for different domains, quality requirements, and training objectives.